# HanuAi ML Assessment — Task 1: Reviews and Sentiment Analysis

## Compliance note
The assignment requires Best Buy Canada reviews and also says not to violate site policy. Best Buy Canada's current Conditions of Use explicitly prohibit scraping and data-mining. Therefore this notebook does **not** automate collection from BestBuy.ca. Obtain written permission or a licensed/authorized export first. Do not bypass CAPTCHA, access controls, rate limits, or blocking. This is a deliberate compliance decision, not a claim that reviews have been scraped.

The remaining cells fully implement cleaning, sentiment scoring, aspect tags, validation, and exports for a lawfully obtained review CSV.

In [ ]:
# Install once if needed:
# %pip install pandas vaderSentiment matplotlib openpyxl
import pandas as pd, numpy as np, re, hashlib
from pathlib import Path
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
INPUT = Path('authorized_reviews.csv')  # Put an authorized review export here
OUTPUT = Path('Task1_Reviews_Sentiment_Results.csv')


In [ ]:
# Expected fields; column matching is case-insensitive and ignores extra spaces.
if not INPUT.exists():
    raise FileNotFoundError('Add an authorized review export named authorized_reviews.csv. No Best Buy data is fabricated.')
reviews = pd.read_csv(INPUT)
reviews.columns = [re.sub(r'\\s+', ' ', c.strip()).lower() for c in reviews.columns]
# Map common source column names to the assessment output schema.
aliases = {
 'title':['title','review title'], 'review text':['review text','review','body','content'],
 'date':['date','review date','reviewed at'], 'rating':['rating','star rating','stars'],
 'source':['source','platform'], 'reviewer name':['reviewer name','reviewer','author','user name'],
 'primary key':['primary key','id','review id','comment id']}
for target, candidates in aliases.items():
    if target not in reviews.columns:
        found = next((c for c in candidates if c in reviews.columns), None)
        reviews[target] = reviews[found] if found else ''


In [ ]:
# Normalize dates and ratings; create a stable ID if the source has no review ID.
reviews['date'] = pd.to_datetime(reviews['date'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('')
reviews['rating'] = pd.to_numeric(reviews['rating'], errors='coerce')
reviews.loc[~reviews['rating'].between(1,5), 'rating'] = np.nan
reviews['review text'] = reviews['review text'].fillna('').astype(str)
reviews['title'] = reviews['title'].fillna('').astype(str)
reviews['source'] = reviews['source'].replace('', 'Authorized review export').fillna('Authorized review export')
reviews['reviewer name'] = reviews['reviewer name'].fillna('').astype(str)
if reviews['primary key'].fillna('').astype(str).str.strip().eq('').any():
    generated = (reviews['title']+'|'+reviews['review text']+'|'+reviews['date']+'|'+reviews['rating'].astype(str)).map(lambda x: hashlib.sha256(x.encode()).hexdigest()[:16])
    reviews['primary key'] = reviews['primary key'].fillna('').astype(str)
    reviews.loc[reviews['primary key'].str.strip().eq(''), 'primary key'] = generated[reviews['primary key'].str.strip().eq('')]
reviews = reviews.drop_duplicates(subset=['primary key']).copy()
print('Rows:',len(reviews),'Unique IDs:',reviews['primary key'].nunique(),'Missing review text:',reviews['review text'].str.strip().eq('').sum())


In [ ]:
# VADER is a transparent baseline for English text, not a perfect product-review model.
analyzer = SentimentIntensityAnalyzer()
def score_review(text):
    compound = analyzer.polarity_scores(str(text))['compound']
    label = 'Positive' if compound >= 0.05 else ('Negative' if compound <= -0.05 else 'Neutral')
    return pd.Series({'Sentiment Score':compound,'Sentiment':label})
combined = (reviews['title'].str.strip()+' . '+reviews['review text'].str.strip()).str.strip(' .')
reviews[['Sentiment Score','Sentiment']] = combined.apply(score_review)


In [ ]:
# Explainable aspect tags; multi-label tagging is allowed.
ASPECTS = {
 'Design & appearance':r'colour|color|design|look|appearance|style',
 'Quality & durability':r'quality|durab|broke|broken|defect|damaged|sturdy',
 'Ease of use':r'easy|simple|setup|install|assembly|instructions',
 'Performance':r'work(s|ed|ing)?|performance|fast|slow|powerful|effective',
 'Value & price':r'price|value|worth|expensive|cheap|cost',
 'Delivery & packaging':r'delivery|shipping|arrived|package|packaging',
 'Customer service':r'customer service|support|return|refund|warranty'
}
def aspect_tags(text):
    low=str(text).lower()
    return '; '.join(k for k,p in ASPECTS.items() if re.search(p,low)) or 'Other/unspecified'
reviews['Aspect Tags'] = combined.apply(aspect_tags)
reviews = reviews.rename(columns={'primary key':'Primary Key','title':'Title','review text':'Review Text','date':'Date','rating':'Rating','source':'Source','reviewer name':'Reviewer Name'})
reviews = reviews[['Primary Key','Title','Review Text','Date','Rating','Source','Reviewer Name','Sentiment Score','Sentiment','Aspect Tags']]
reviews.to_csv(OUTPUT,index=False)
display(reviews.head(10))
display(reviews['Sentiment'].value_counts().rename_axis('Sentiment').to_frame('Count'))
print('Saved:',OUTPUT)


## Scraping challenges and responsible handling
- **IP blocking / throttling:** do not rotate proxies to evade a site's restriction. Stop automation, reduce request volume, and request written permission or an approved data feed.
- **CAPTCHA / access denial:** do not bypass. Stop and use the site's authorized access route.
- **Transient connection errors:** only for an authorized endpoint, use bounded exponential backoff (for example 1, 2, 4 seconds; limited retries) and log failures.
- **Pagination and sorting:** implement only after permission and after confirming the site's documented interface/API. Record which sort/filter was applied and deduplicate across pages.
- **Data quality:** validate dates, ratings in [1,5], unique IDs, missing review text, and manually label a sample to evaluate sentiment accuracy.

## Reporting guidance
Report sentiment proportions and aspect-tag counts only after loading real authorized reviews. A high positive score can reflect tone rather than verified product quality. Review negative examples manually before recommending product changes. Best Buy Canada Conditions of Use: https://www.bestbuy.ca/en-ca/help/policies-and-terms-and-conditions/conditions-of-use